In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
df1 = pd.read_csv('ASQP_MA/datasets/dataset.csv')
df2 = pd.read_json('datasets/annotated.jsonl', lines=True)

col1 = df1['Comment Sentiment']    # Original sentiment from SMILE-College dataset
col2 = df2['cats'].apply(lambda x: x[0].upper() if x and len(x) > 0 else None)  # Processed sentiment from annotated dataset, skip if 'cats' is empty

# Calculate the matching ratio between the two sentiment columns
result = col1 == col2
print('Matched samples: ', result.sum())
print('Total samples: ', len(result))
print('Matching ratio: ', result.sum() / len(result))

### Mismatched sample analysis

In [5]:
# List mismatched samples
filtered = result[result == False]
print('Mismatched samples: ', filtered.index.tolist())

In [6]:
df2 = pd.read_json('datasets/annotated_revised.jsonl', lines=True)
col2 = df2['cats'].apply(lambda x: x[0].upper())    # Processed sentiment from annotated dataset

# Calculate the matching ratio between the two sentiment columns
result = col1 == col2
print('Matched samples: ', result.sum())
print('Total samples: ', len(result))
print('Matching ratio: ', result.sum() / len(result))

### Overall Sentiment Statistics

In [8]:
# Overall sentiment counts — publication-quality horizontal bar and save as PDF
sentiments = df1['Comment Sentiment']
# Count the occurrences of each sentiment
sentiment_counts = sentiments.value_counts()

import matplotlib as mpl
mpl.rcParams.update({'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 14})
import seaborn as sns
sns.set_style('white')

fig, ax = plt.subplots(figsize=(10, 3))
ax = sns.barplot(y=sentiment_counts.index, x=sentiment_counts.values, palette='viridis', ax=ax)

# Add numbers on each bar (right side)
max_val = sentiment_counts.values.max() if len(sentiment_counts)>0 else 0
offset = max(1, int(max_val*0.01))
for i, v in enumerate(sentiment_counts.values):
    ax.text(v + offset, i, str(int(v)), va='center', fontsize=10)

ax.set_ylabel('Overall Sentiment', fontsize=14)
ax.set_xlabel('Number of Samples', fontsize=13)

# Ensure axis ticks and labels are visible and styled
ax.tick_params(axis='y', labelsize=12)
ax.tick_params(axis='x', labelsize=12)
ax.xaxis.set_visible(True)
ax.yaxis.set_visible(True)

# Ensure left and bottom spines are visible, remove top/right
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Turn off grid for publication look
ax.grid(False)

sns.despine(left=False, bottom=False)
plt.tight_layout()

# Save as PDF for paper
output_path = 'outputs/sentiment_counts_horizontal.pdf'
fig.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
print('Saved sentiment counts to', output_path)
fig

### Aspects Statistics

In [12]:
f2 = pd.read_csv('/home/txie/Mental_Health/SMILE-College_Aspect_Analysis/datasets/majority_voting_results.csv')
annotations = df2['merged_entities'].apply(eval).tolist()
stats = {
    'On-campus Service': {'Positive': 0, 'Neutral': 0, 'Negative': 0},
    'Counseling Service': {'Positive': 0, 'Neutral': 0, 'Negative': 0},
    'Mental Health Service': {'Positive': 0, 'Neutral': 0, 'Negative': 0},
    'Wellness Service': {'Positive': 0, 'Neutral': 0, 'Negative': 0},
    'Therapy Service': {'Positive': 0, 'Neutral': 0, 'Negative': 0},
    'Hotline Service': {'Positive': 0, 'Neutral': 0, 'Negative': 0},
    'Service Availability': {'Positive': 0, 'Neutral': 0, 'Negative': 0},
    'General': {'Positive': 0, 'Neutral': 0, 'Negative': 0}
}

for annotation in annotations:
    for span in annotation:
        aspects = span['aspects']
        sentiments = span['sentiments']
        for i in range(len(aspects)):
            aspect = aspects[i]
            sentiment = sentiments[i]
            stats[aspect][sentiment] += 1

In [13]:
df_stats = []
for aspect, sentiments in stats.items():
    for sentiment, count in sentiments.items():
        df_stats.append({'Aspect': aspect, 'Sentiment': sentiment, 'Count': count})

df_stats = pd.DataFrame(df_stats)

import matplotlib as mpl
mpl.rcParams.update({'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 14})
import seaborn as sns
sns.set_style('white')

fig, ax = plt.subplots(figsize=(12, 9))

# Horizontal grouped bar plot: aspects on y, counts on x, hue by sentiment
palette = {'Positive': 'tab:blue', 'Neutral': 'tab:gray', 'Negative': 'tab:orange'}
ax = sns.barplot(
    data=df_stats,
    y='Aspect',
    x='Count',
    hue='Sentiment',
    palette=palette,
    ax=ax
)

# Remove gridlines for publication look
ax.grid(False)

# Annotate numbers on bars
for p in ax.patches:
    width = p.get_width()
    if width > 0:
        ax.annotate(f'{int(width)}', (width + max(1, int(width*0.01)), p.get_y() + p.get_height() / 2),
                    va='center', fontsize=10)

# Labels and styling
ax.set_ylabel('High-level Aspect Category', fontsize=14)
ax.set_xlabel('Number of Samples', fontsize=13)
plt.setp(ax.get_yticklabels(), fontsize=13)

# Position legend and remove frame
leg = ax.legend(title='Sentiment', loc='upper right')
leg.get_frame().set_linewidth(0.0)

# Ensure left and bottom spines are visible, remove top/right for cleaner look
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

sns.despine(trim=False)
plt.tight_layout()

# Save as PDF
output_path = 'outputs/aspect_sentiment_counts.pdf'
fig.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
print('Saved aspect sentiment bar plot to', output_path)
fig